<a href="https://colab.research.google.com/github/abdelrahman-rajab-hassan/abdelrahman-rajab-hassan/blob/main/Ensemble_Trees_(Core).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ensemble Tree Methods for ``Housing Price Prediction``
---

## 📌 Project Overview
This notebook focuses on improving predictive results for the **Boston Housing Dataset** by transitioning from simple regression trees to more robust **ensemble tree models**.

### 🛠 Model Specifications
* **Target Variable:** `PRICE`
* **Data Status:** All numeric and pre-cleaned.
* **Preprocessing Note:** Scaling is **not required** for tree-based ensemble methods.
* **Mandatory Step:** Perform a **Train/Test Split** prior to model training.

---

## 🎯 Primary Objective
> **Task:** Develop and tune the best possible model to predict house prices with maximum accuracy.

---

In [ ]:
# import all necessary libraries, functions and classes
import numpy as np
import pandas as pd


from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

from sklearn import set_config
from google.colab import drive

drive.mount('/content/drive')
set_config(transform_output = 'pandas')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Helper function to compute, print, and tabulate regression performance metrics for model evaluation.

def regression_metrics(y_true, y_pred, label='', verbose=True, output_dict=False):
    # Get metrics
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    r_squared = r2_score(y_true, y_pred)

    if verbose == True:
        # Print Result with Label and Header
        header = "-"*60
        print(header, f"Regression Metrics: {label}", header, sep='\n')
        print(f"- MAE = {mae:,.3f}")
        print(f"- MSE = {mse:,.3f}")
        print(f"- RMSE = {rmse:,.3f}")
        print(f"- R^2 = {r_squared:,.3f}")

    if output_dict == True:
        metrics = {'Label': label, 'MAE': mae,
                   'MSE': mse, 'RMSE': rmse, 'R^2': r_squared}
        return metrics

def evaluate_regression(reg, X_train, y_train, X_test, y_test, verbose=True,
                        output_frame=False):
    # Get predictions for training data
    y_train_pred = reg.predict(X_train)

    # Call the helper function to obtain regression metrics for training data
    results_train = regression_metrics(y_train, y_train_pred, verbose=verbose,
                                       output_dict=output_frame,
                                       label='Training Data')
    print()

    # Get predictions for test data
    y_test_pred = reg.predict(X_test)

    # Call the helper function to obtain regression metrics for test data
    results_test = regression_metrics(y_test, y_test_pred, verbose=verbose,
                                      output_dict=output_frame,
                                      label='Test Data')

    # Store results in a dataframe if output_frame is True
    if output_frame:
        results_df = pd.DataFrame([results_train, results_test])
        # Set the label as the index
        results_df = results_df.set_index('Label')
        # Set index.name to none to get a cleaner looking result
        results_df.index.name = None
        # Return the dataframe
        return results_df.round(3)

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/AXSOSACADEMY/AXSOSACADEMY/02-IntroML/Week06/Data/Boston_Housing_from_Sklearn - Boston_Housing_from_Sklearn.csv')
df.head()

,CRIM,NOX,RM,AGE,PTRATIO,LSTAT,PRICE
0,0.00632,0.538,6.575,65.2,15.3,4.98,24.0
1,0.02731,0.469,6.421,78.9,17.8,9.14,21.6
2,0.02729,0.469,7.185,61.1,17.8,4.03,34.7
3,0.03237,0.458,6.998,45.8,18.7,2.94,33.4
4,0.06905,0.458,7.147,54.2,18.7,5.33,36.2


In [ ]:
# check for null values
df.isna().sum().sum()

np.int64(0)

In [ ]:
# check for duplicates
df[df.duplicated()].sum().sum()

np.float64(0.0)

In [ ]:
# train, test and split the data
y = df['PRICE']

X = df.drop(columns = 'PRICE')

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42)

In [ ]:
# run the head() function
X_train.head()

,CRIM,NOX,RM,AGE,PTRATIO,LSTAT
182,0.09103,0.4880,7.155,92.2,17.8,4.82
155,3.53501,0.8710,6.152,82.6,14.7,15.02
280,0.03578,0.4429,7.820,64.5,14.9,3.76
126,0.38735,0.5810,5.613,95.6,19.1,27.26
329,0.06724,0.4600,6.333,17.2,16.9,7.34


### Train and evaluate the defualt values of ``Bagged Tree``



In [ ]:
# instantiate the BaggingRegressor function before applying GridSearchCV
bagging_tree = BaggingRegressor(random_state = 42)
bagging_tree

BaggingRegressor(random_state=42)

In [ ]:
bagging_tree.fit(X_train, y_train)

BaggingRegressor(random_state=42)

In [ ]:
# run the evaluate function for the default values of Bagged Tree
evaluate_regression(bagging_tree, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 1.103
- MSE = 3.487
- RMSE = 1.867
- R^2 = 0.961

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 2.316
- MSE = 12.575
- RMSE = 3.546
- R^2 = 0.820


### Use GridSearchCV Wrappter to optimize performance on the test for the ``Bagged Tree``



In [ ]:
# get the hyperparameters of the bagged tree
bagging_tree.get_params()

{'bootstrap': True,
 'bootstrap_features': False,
 'estimator': None,
 'max_features': 1.0,
 'max_samples': 1.0,
 'n_estimators': 10,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [ ]:
# grid params
param_grid = {
    'n_estimators': [10, 50, 100, 200],
    'max_samples': [0.5, 0.7, 1.0],
    'max_features': [0.5, 0.7, 1.0],
    'bootstrap': [True, False],
    'bootstrap_features': [True, False]
}

In [ ]:
grid = GridSearchCV(bagging_tree, param_grid, cv = 3, n_jobs = -1)
grid.fit(X_train, y_train)

GridSearchCV(cv=3, estimator=BaggingRegressor(random_state=42), n_jobs=-1,
             param_grid={'bootstrap': [True, False],
                         'bootstrap_features': [True, False],
                         'max_features': [0.5, 0.7, 1.0],
                         'max_samples': [0.5, 0.7, 1.0],
                         'n_estimators': [10, 50, 100, 200]})

In [ ]:
best_bagging_tree = grid.best_estimator_
best_bagging_tree

BaggingRegressor(bootstrap_features=True, n_estimators=200, random_state=42)

In [ ]:
# re-run the evaluate function for the best result values of Bagged Tree
evaluate_regression(best_bagging_tree, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 0.944
- MSE = 2.029
- RMSE = 1.425
- R^2 = 0.977

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 2.277
- MSE = 12.748
- RMSE = 3.570
- R^2 = 0.818


### Train and evaluate the default values of ``RandomForestRegressor``

In [ ]:
# Instantiate the RandomForestRegressor with default values
random_forest_reg = RandomForestRegressor(random_state = 42)
random_forest_reg

RandomForestRegressor(random_state=42)

In [ ]:
# Fit the default RandomForestRegressor
random_forest_reg.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [ ]:
# Evaluate the default RandomForestRegressor
evaluate_regression(random_forest_reg, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 0.954
- MSE = 2.028
- RMSE = 1.424
- R^2 = 0.977

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 2.208
- MSE = 11.635
- RMSE = 3.411
- R^2 = 0.834


### Use GridSearchCV Wrapper to optimize performance for ``RandomForestRegressor``

In [ ]:
# Get the hyperparameters of the RandomForestRegressor
random_forest_reg.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 3, 5],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', 0.5],
    'max_samples': [0.6, 0.8, 1.0],
}

In [ ]:
# Instantiate and fit GridSearchCV for RandomForestRegressor
grid_rf = GridSearchCV(random_forest_reg, param_grid_rf, cv = 3, n_jobs = -1)
grid_rf.fit(X_train, y_train)

GridSearchCV(cv=3, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'bootstrap': [True, False],
                         'max_depth': [None, 10, 20],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]})

In [ ]:
# Get the best estimator from GridSearchCV
best_random_forest = grid_rf.best_estimator_
best_random_forest

RandomForestRegressor(max_depth=20, n_estimators=200, random_state=42)

In [ ]:
# Evaluate the best RandomForestRegressor
evaluate_regression(best_random_forest, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 0.963
- MSE = 2.119
- RMSE = 1.456
- R^2 = 0.976

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 2.185
- MSE = 11.668
- RMSE = 3.416
- R^2 = 0.833


5. ``Which model and model parameters provided the best results?``

  * The default RandomForestRegressor provided the best results with a test R² of 0.834 and the lowest RMSE of 3.411 and MAE of 2.208 — slightly edging out the tuned version. The default parameters (n_estimators=100, max_features=1.0, max_depth=None) were already well-suited to this data.

---

6. ``xplain in a text cell how your model will perform if deployed by referring
to the metrics. Ex. How close can your stakeholders expect its predictions
to be to the true value? ``

  * With a test R² of 0.834, the model explains about 83% of the variance in the target variable. Stakeholders can expect predictions to be off by roughly ±2.2 units on average (MAE), and in worse cases up to ±3.4 units (RMSE). The gap between training R² (0.977) and test R² (0.834) indicates some overfitting, meaning real-world performance may be slightly lower than what training suggests — but the test metrics are the honest estimate to rely on.